# Movie Recommendation Engine - Production Model

Production-ready Item-Based Collaborative Filtering model with support for new user recommendations.

**Model Performance:**
- RMSE: 0.8492
- MAE: 0.6501
- Dataset: MovieLens 100K

## 1. Setup and Load Dependencies

In [ ]:
import pandas as pd
import numpy as np
import pickle
import json
from pathlib import Path

print("Libraries loaded successfully!")

## 2. Load Pre-trained Model and Data

In [ ]:
# Load the pre-trained item-based collaborative filtering model
model_dir = Path('../movie-engine-data/models')
data_dir = Path('../movie-engine-data/processed/ml-100k')

# Load item similarity matrix
with open(model_dir / 'item_similarity_matrix.pkl', 'rb') as f:
    item_similarity_df = pickle.load(f)

# Load user-item matrix
with open(model_dir / 'user_item_matrix.pkl', 'rb') as f:
    user_item_matrix = pickle.load(f)

# Load model metadata
with open(model_dir / 'item_based_metadata.json', 'r') as f:
    model_metadata = json.load(f)

# Load movie and ratings data
movies_df = pd.read_csv(data_dir / 'movies.csv')
ratings_df = pd.read_csv(data_dir / 'ratings.csv')

print("Model loaded successfully!")
print(f"Model type: {model_metadata['model_type']}")
print(f"Item similarity matrix shape: {item_similarity_df.shape}")
print(f"User-item matrix shape: {user_item_matrix.shape}")
print(f"Performance - RMSE: {model_metadata['rmse']}, MAE: {model_metadata['mae']}")

## 3. Core Prediction Functions

In [ ]:
def predict_rating_for_new_user(movie_id, user_ratings, item_similarity_df, k=10):
    """
    Predict rating for a specific movie based on new user's ratings
    
    Parameters:
    - movie_id: ID of movie to predict rating for
    - user_ratings: Dict of {movieId: rating} for movies the user has rated
    - item_similarity_df: Pre-trained item similarity matrix
    - k: Number of similar items to consider
    
    Returns:
    - Predicted rating (float)
    """
    if movie_id not in item_similarity_df.columns:
        return 3.0  # Default neutral rating
    
    # Get similarity to movies the user has rated
    similarities = item_similarity_df[movie_id]
    
    weighted_sum = 0
    similarity_sum = 0
    
    # Calculate weighted average based on similar movies user has rated
    for rated_movie_id, rating in user_ratings.items():
        if rated_movie_id in similarities.index:
            sim = similarities[rated_movie_id]
            weighted_sum += sim * rating
            similarity_sum += abs(sim)
    
    if similarity_sum == 0:
        return 3.0
    
    predicted_rating = weighted_sum / similarity_sum
    return np.clip(predicted_rating, 0.5, 5.0)


def get_popular_movies(movies_df, ratings_df, n=10):
    """
    Get popular movies based on average rating and rating count
    
    Parameters:
    - movies_df: Movies dataframe
    - ratings_df: Ratings dataframe
    - n: Number of movies to return
    
    Returns:
    - DataFrame with popular movies
    """
    popular = ratings_df.groupby('movieId').agg({
        'rating': ['mean', 'count']
    }).reset_index()
    popular.columns = ['movieId', 'avg_rating', 'rating_count']
    
    # Filter for movies with at least 50 ratings
    popular = popular[popular['rating_count'] >= 50]
    popular = popular.sort_values('avg_rating', ascending=False).head(n)
    
    return movies_df[movies_df['movieId'].isin(popular['movieId'])][['movieId', 'title']].merge(
        popular[['movieId', 'avg_rating']], on='movieId'
    ).rename(columns={'avg_rating': 'predicted_rating'})


print("Core prediction functions loaded!")

## 4. Recommendation Engine

In [ ]:
def recommend_movies(user_ratings, item_similarity_df, movies_df, ratings_df, n=1, k=10):
    """
    Generate movie recommendations using hybrid onboarding strategy
    
    Strategy:
    - No ratings: Return popular movies
    - Few ratings (1-4): Blend personalized + popular recommendations
    - Sufficient ratings (5+): Fully personalized item-based CF
    
    Parameters:
    - user_ratings: Dict of {movieId: rating} for movies the user has rated
    - item_similarity_df: Pre-trained item similarity matrix
    - movies_df: Movies dataframe
    - ratings_df: Ratings dataframe (for popular fallback)
    - n: Number of recommendations to return
    - k: Number of similar items to consider in predictions
    
    Returns:
    - DataFrame with recommended movies and predicted ratings
    """
    num_ratings = len(user_ratings)
    
    # Case 1: No ratings - recommend popular movies
    if num_ratings == 0:
        print("No ratings provided. Returning popular movies.")
        return get_popular_movies(movies_df, ratings_df, n)
    
    # Get all available movies
    rated_movie_ids = list(user_ratings.keys())
    all_movie_ids = item_similarity_df.columns.tolist()
    unrated_movies = [m for m in all_movie_ids if m not in rated_movie_ids]
    
    # Generate personalized predictions
    predictions = []
    for movie_id in unrated_movies:
        predicted_rating = predict_rating_for_new_user(movie_id, user_ratings, item_similarity_df, k)
        predictions.append({
            'movieId': movie_id,
            'predicted_rating': predicted_rating
        })
    
    predictions_df = pd.DataFrame(predictions)
    
    if len(predictions_df) == 0:
        return get_popular_movies(movies_df, ratings_df, n)
    
    # Case 2: Few ratings (1-4) - blend personalized + popular
    if num_ratings < 5:
        print(f"{num_ratings} rating(s) provided. Blending personalized and popular recommendations.")
        
        # Get top personalized recommendations
        personalized = predictions_df.sort_values('predicted_rating', ascending=False).head(n * 2)
        personalized = movies_df[['movieId', 'title']].merge(personalized, on='movieId')
        
        # Get popular recommendations
        popular = get_popular_movies(movies_df, ratings_df, n * 2)
        
        # Combine and deduplicate
        combined = pd.concat([personalized, popular]).drop_duplicates('movieId')
        return combined.head(n)
    
    # Case 3: Sufficient ratings (5+) - fully personalized
    print(f"{num_ratings} ratings provided. Generating fully personalized recommendations.")
    predictions_df = predictions_df.sort_values('predicted_rating', ascending=False).head(n)
    recommendations = movies_df[['movieId', 'title']].merge(predictions_df, on='movieId')
    
    return recommendations


print("Recommendation engine loaded!")

## 5. Production API Function

In [ ]:
def get_recommendations(user_ratings, n=10, k=10):
    """
    Main API function to get movie recommendations
    
    Parameters:
    - user_ratings: Dict of {movieId: rating} for movies the user has rated
                   Can be empty dict for new users with no ratings
    - n: Number of recommendations to return (default: 10)
    - k: Number of similar items to consider (default: 10)
    
    Returns:
    - DataFrame with columns: movieId, title, predicted_rating
    
    Example:
    >>> user_ratings = {1: 5.0, 50: 4.5, 32: 2.0}
    >>> recommendations = get_recommendations(user_ratings, n=5)
    """
    return recommend_movies(
        user_ratings=user_ratings,
        item_similarity_df=item_similarity_df,
        movies_df=movies_df,
        ratings_df=ratings_df,
        n=n,
        k=k
    )


print("Production API ready!")
print("\nUsage: get_recommendations(user_ratings, n=10, k=10)")
print("  - user_ratings: dict of {movieId: rating}")
print("  - n: number of recommendations (default: 10)")
print("  - k: number of neighbors to consider (default: 10)")

## 6. Test Cases

In [ ]:
# Test Case 1: New user with no ratings
print("=" * 80)
print("TEST 1: New user with no ratings (Cold start)")
print("=" * 80)

new_user_ratings = {}
recommendations = get_recommendations(new_user_ratings, n=5)
print("\nRecommendations:")
print(recommendations.to_string(index=False))

In [ ]:
# Test Case 2: User with a few ratings
print("=" * 80)
print("TEST 2: User with 3 ratings (Hybrid approach)")
print("=" * 80)

few_ratings = {
    1: 5.0,      # Toy Story - liked it
    50: 3.0,     # Usual Suspects - meh
    32: 1.0      # 12 Monkeys - didn't like
}

print("\nUser's ratings:")
for movie_id, rating in few_ratings.items():
    title = movies_df[movies_df['movieId'] == movie_id]['title'].values[0]
    print(f"  {title}: {rating}")

recommendations = get_recommendations(few_ratings, n=5)
print("\nRecommendations:")
print(recommendations.to_string(index=False))

In [ ]:
# Test Case 3: User with sufficient ratings for personalized recommendations
print("=" * 80)
print("TEST 3: User with 7 ratings (Fully personalized)")
print("=" * 80)

sufficient_ratings = {
    1: 5.0,      # Toy Story
    50: 3.0,     # Usual Suspects
    32: 1.0,      # 12 Monkeys
    110: 5.0,    # Braveheart
    260: 5.0,    # Star Wars: Episode IV
    356: 3.0,    # Forrest Gump
    296: 5.0     # Pulp Fiction
}

print("\nUser's ratings:")
for movie_id, rating in sufficient_ratings.items():
    title = movies_df[movies_df['movieId'] == movie_id]['title'].values[0]
    print(f"  {title}: {rating}")

recommendations = get_recommendations(sufficient_ratings, n=10)
print("\nRecommendations:")
print(recommendations.to_string(index=False))

## 7. Helper Function: Find Movie by Title

In [ ]:
def find_movie(title_search):
    """
    Search for movies by title (partial match)
    
    Parameters:
    - title_search: String to search for in movie titles
    
    Returns:
    - DataFrame with matching movies
    """
    matches = movies_df[movies_df['title'].str.contains(title_search, case=False, na=False)]
    return matches[['movieId', 'title']]


# Example usage
print("Search for movies containing 'Star Wars':")
print(find_movie('Star Wars').to_string(index=False))

## 8. Interactive Demo

Use this cell to test with your own movie ratings!

In [ ]:
# ============================================
# CUSTOMIZE YOUR RATINGS HERE
# ============================================
# Use find_movie() above to search for movie IDs
# Rating scale: 0.5 to 5.0

my_ratings = {
    # Add your ratings here
    # movieId: rating
}

# Get recommendations
if len(my_ratings) > 0:
    print("Your ratings:")
    for movie_id, rating in my_ratings.items():
        title = movies_df[movies_df['movieId'] == movie_id]['title'].values[0]
        print(f"  {title}: {rating}")
    
    print("\n" + "=" * 80)
    my_recommendations = get_recommendations(my_ratings, n=10)
    print("\nYour personalized recommendations:")
    print(my_recommendations.to_string(index=False))
else:
    print("Add your ratings to the 'my_ratings' dictionary above!")
    print("Example: my_ratings = {1: 5.0, 50: 4.0}")

## 9. Model Information

In [ ]:
# Display complete model information
print("=" * 80)
print("MODEL INFORMATION")
print("=" * 80)
print(f"Model Type: {model_metadata['model_type']}")
print(f"Dataset: {model_metadata['dataset']}")
print(f"Created: {model_metadata['created_date']}")
print(f"\nPerformance Metrics:")
print(f"  RMSE: {model_metadata['rmse']}")
print(f"  MAE: {model_metadata['mae']}")
print(f"\nModel Configuration:")
print(f"  Number of users: {model_metadata['num_users']}")
print(f"  Number of movies: {model_metadata['num_movies']}")
print(f"  K-neighbors: {model_metadata['k_neighbors']}")
print(f"\nMatrix Shapes:")
print(f"  Item similarity: {model_metadata['matrix_shape']}")
print(f"\nRecommendation Strategy:")
print(f"  0 ratings: Popular movies")
print(f"  1-4 ratings: Hybrid (personalized + popular)")
print(f"  5+ ratings: Fully personalized item-based CF")
print("=" * 80)